# Learning profiles: paired-data cleaning that remembers



Phase 4 adds `fd.learn()`: instead of hand-writing cleaning rules, you show freshdata a

**messy/clean pair** of the same rows and it learns a reusable `.fdprofile` — a set of

typed rules, value maps, and (optionally) retrieval examples that later replay against new,

similarly-shaped data via `fd.clean(df, profile=...)`.



This notebook runs a small but realistic example end to end: an ecommerce customer table

from India with messy emails, phone formats, day-first dates, currency strings, status

typos, and a nonstandard missing-value sentinel.

## 1. A messy/clean pair



We build 30 rows of "messy" input and the corresponding "clean" ground truth. Two status

typos — `"Shipped"` and `"Deliverd"` — each occur with support 5, right at the default

`min_support=5` gate, so both should be learned as value-map rules rather than demoted to

suggest-only examples.

In [ ]:
import pandas as pd



import freshdata as fd



n = 30

cust_id = list(range(1, n + 1))



# email: every 3rd row upper-cased, every 3rd row padded with whitespace

emails_clean = [f"user{i}@example.in" for i in range(1, n + 1)]

emails_messy = [

    e.upper() if i % 3 == 0 else (f"  {e} " if i % 3 == 1 else e)

    for i, e in enumerate(emails_clean)

]



# phone: rotate through a few common messy IN formats

phones_clean = [f"+9198765{43200 + i:05d}" for i in range(n)]

phones_messy = []

for i, p in enumerate(phones_clean):

    digits = p.replace("+91", "")

    style = i % 4

    if style == 0:

        phones_messy.append(f"0{digits}")

    elif style == 1:

        phones_messy.append(f"{digits[:5]} {digits[5:]}")

    elif style == 2:

        phones_messy.append(f"91-{digits}")

    else:

        phones_messy.append(p)



# signup_date: half day-first, half already ISO

dates_clean = [f"2024-{1 + (i % 9):02d}-{10 + (i % 15):02d}" for i in range(n)]

dates_messy = [

    f"{d.split('-')[2]}/{d.split('-')[1]}/{d.split('-')[0]}" if i % 2 == 0 else d

    for i, d in enumerate(dates_clean)

]



# amount_spent: rupee-formatted strings -> floats

amounts_clean = [round(500 + i * 37.5, 2) for i in range(n)]

amounts_messy = [f"₹{a:,.2f}" for a in amounts_clean]



# status: two typo'd spellings, each with support >= min_support (5)

status_clean = (['shipped'] * 10) + (['delivered'] * 10) + (['cancelled'] * 10)

status_messy = (

    (['Shipped'] * 5 + ['shipped'] * 5)

    + (['Deliverd'] * 5 + ['delivered'] * 5)

    + (['cancelled'] * 10)

)



# country: a nonstandard sentinel for missing, every 6th row

country_clean = ['India'] * n

country_messy = ['India'] * n

for i in range(0, n, 6):

    country_messy[i] = 'N/A'

    country_clean[i] = None



messy_df = pd.DataFrame(

    {

        "cust_id": cust_id,

        "email": emails_messy,

        "phone": phones_messy,

        "signup_date": dates_messy,

        "amount_spent": amounts_messy,

        "status": status_messy,

        "country": country_messy,

    }

)

clean_df = pd.DataFrame(

    {

        "cust_id": cust_id,

        "email": emails_clean,

        "phone": phones_clean,

        "signup_date": dates_clean,

        "amount_spent": amounts_clean,

        "status": status_clean,

        "country": country_clean,

    }

)



messy_df.head()

## 2. Learn a profile



`fd.learn` runs the 5-stage pipeline (align → diff → classify → extract → fit/holdout-validate)

and returns a `LearningProfile`. `context=` is free text that seeds semantic-type inference,

same as `fd.clean(df, context=...)`. `key="cust_id"` aligns messy/clean rows unambiguously.

In [ ]:
profile = fd.learn(

    messy_df,

    clean_df,

    context="Ecommerce customer data from India",

    key="cust_id",

)

print(profile.summary())

print("contains_raw_values:", profile.manifest.contains_raw_values)

`contains_raw_values` is `False`: by default (`privacy="mask"`), sensitive semantic types

(`email`, `phone`, `person_name`, `national_id`, `address`, `postal_code`, `free_text`) are

HMAC-masked before anything is stored, and no raw literal imputation values are ever learned.

Only the non-sensitive `status` column contributes real value-map literals here (`"Shipped"

→ "shipped"`, `"Deliverd" → "delivered"`), because `status` isn't a sensitive semantic type.

## 3. Save, and confirm no literal PII leaks into the file



A `.fdprofile` is a zip with a manifest that hashes every member for integrity. We check the

raw bytes of the saved file for any of the real emails or phone numbers from the training

data — none should appear, since those columns are masked by default.

In [ ]:
fd.save_profile(profile, "/tmp/e2e_orders.fdprofile")



with open("/tmp/e2e_orders.fdprofile", "rb") as f:

    raw_bytes = f.read()



leak_terms = emails_clean + phones_clean

leaks = [t for t in leak_terms if t.encode("utf-8") in raw_bytes]

print("literal leaks found in zip bytes:", leaks)

## 4. Load the profile and replay it on new data



`fd.clean(df, profile=..., return_report=True)` returns a `(cleaned_df, report)` tuple.

Profile-backed proposals are gathered by the semantic layer, which stays off by default — pass

`semantic_mode="auto"` (or `"assist"`/`"review"`) explicitly, exactly as you would to enable

semantic assistance without a profile.

In [ ]:
loaded = fd.load_profile("/tmp/e2e_orders.fdprofile")



new_df = pd.DataFrame(

    {

        "cust_id": [1001, 1002, 1003, 1004],

        "email": [

            "  NEW.USER@example.in ",

            "another@example.in",

            "THIRD@EXAMPLE.IN",

            "fourth@example.in ",

        ],

        "phone": ["098765 43299", "91-9876543298", "+919876543297", "09876543296"],

        "signup_date": ["15/03/2024", "2024-03-16", "20/03/2024", "2024-03-18"],

        "amount_spent": ["₹1,200.00", "₹950.50", "₹2,000.00", "₹800.00"],

        "status": ["Shipped", "Deliverd", "cancelled", "Shipped"],

        "country": ["India", "N/A", "India", "India"],

    }

)

new_df

### Baseline: `fd.clean(new_df)` with no profile



General-purpose heuristics (phone/date/currency normalization, sentinel-to-missing) still

apply without any profile. But `"Shipped"` and `"Deliverd"` aren't recognizable as typos by

generic rules — there's no dictionary that says "Deliverd" means "delivered" for this

dataset's vocabulary. Without the learned value map, both typos pass through unfixed.

In [ ]:
baseline_cleaned, _ = fd.clean(new_df, return_report=True)

baseline_cleaned[["cust_id", "status"]]

### With the learned profile: `fd.clean(new_df, profile=loaded, semantic_mode="auto")`



Now the learned `status` value map replays: both `"Shipped"` and `"Deliverd"` are corrected,

each recorded as a `profile_influenced` action in the report.

In [ ]:
cleaned, report = fd.clean(

    new_df, profile=loaded, semantic_mode="auto", return_report=True

)

cleaned[["cust_id", "status"]]

In [ ]:
actions = [a for a in report.actions if a.metadata.get("profile_influenced")]

print("profile_influenced actions:", len(actions))

for a in actions:

    print(a.column, a.metadata)

## 5. Audit the profile



`profile.audit()` reports every rule the profile carries and what happened to any candidate

that didn't make it into an auto-applied rule. Here, an `email_normalize` rule was proposed

during extraction but **demoted to suggest-only** because its holdout precision (measured on

a held-out split of the training pairs) fell below the default `min_precision=0.98` gate —

the fit/holdout stage is what keeps a profile from replaying a rule that doesn't generalize.

In [ ]:
audit = profile.audit()

print(f"rule_count={audit.rule_count} rules_by_family={audit.rules_by_family}")

print(f"value_map_entries={audit.value_map_entries} masked_entries={audit.masked_entries}")

print(f"demotions={len(audit.demotions)}")

for d in audit.demotions:

    print(" ", d.column, d.family, d.outcome, "-", d.reason)

## Recap



- `fd.learn(messy, clean, context=..., key=...)` learns a `LearningProfile` from a paired

  example, not from hand-written rules.

- Sensitive columns are masked by default; no full rows, no raw sensitive literals, and no

  literal imputation maps are ever stored.

- Low-support or low-precision candidates are demoted to suggest-only rather than silently

  applied — a profile can only replay what it validated on held-out data.

- Replay is opt-in and additive: `fd.clean(df, profile=profile, semantic_mode="auto")` only

  changes behavior when you ask for it; `profile=None` (the default) is byte-identical to

  today's `fd.clean`.

- See `docs/learning-profiles.md` for the full format, privacy model, and CLI

  (`freshdata learn`, `clean --profile`, `profile audit|diff|merge`).